# LAB: Practice with Tavily + LangChain Agents

### Objective
Reinforce your understanding of LangChain agents integrated with Tavily for real-time web search by solving two open-ended tasks. You'll:
- Load and configure a LangChain agent with Tavily
- Build effective prompts
- Generate structured, useful outputs from real-time data

### Setup

In [4]:
from  dotenv import load_dotenv
import warnings
warnings.filterwarnings('ignore')
load_dotenv()

True

In [3]:
from langchain_classic import hub
from langchain_classic.agents import AgentExecutor, create_react_agent
from langchain_core.tools import Tool
import os
import tiktoken
from langchain_community.tools.tavily_search import TavilySearchResults
from langchain_openai import ChatOpenAI


In [10]:
#import os
#from langchain.agents import initialize_agent, AgentType, Tool
#from langchain.chat_models import ChatOpenAI
#from langchain.tools.tavily_search import TavilySearchResults
#import tiktoken
from IPython.display import Markdown, display
#from  dotenv import load_dotenv
import warnings
warnings.filterwarnings('ignore')
#from langchain.tools.tavily_search import TavilySearchResults

load_dotenv()
tavily_search = TavilySearchResults()

# LLM + Encoding
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
encoding = tiktoken.encoding_for_model("gpt-4o-mini")

# Safe Tavily wrapper
def safe_search(query: str) -> str:
    result = tavily_search.run(query)

    # Ensure result is a string — Tavily returns dict with 'snippets' sometimes
    if isinstance(result, dict):
        result_text = result.get("content", "") or str(result)
    else:
        result_text = str(result)

    tokens = encoding.encode(result_text)
    trimmed = encoding.decode(tokens[:3500])  # leave room for GPT-4 response
    return trimmed

prompt = hub.pull("hwchase17/react")
tools = [Tool(name="TavilySafeSearch", func=safe_search, description="Web search tool")]
# Create the agent
agent = create_react_agent(llm, tools, prompt)

# Create agent executor
agent_executor = AgentExecutor(
    agent=agent,
    tools=tools,
    verbose=True,
    handle_parsing_errors=True
)

# Run the agent
response = agent_executor.invoke({"input": "What are the latest AI research on retrieval-augmented generation?"})
print(response["output"])


# LangChain tool
#
#result = tavily_search.invoke({"query": "Latest AI research on retrieval-augmented generation"})
#agent = initialize_agent(tools=tools, llm=llm, agent=AgentType.ZERO_SHOT_REACT_DESCRIPTION, verbose=True)



> Entering new AgentExecutor chain...
I need to find the latest research on retrieval-augmented generation in AI. This involves searching for recent publications or articles that discuss advancements in this area. 
Action: TavilySafeSearch
Action Input: "latest AI research on retrieval-augmented generation 2023"[{'title': 'Latest Advancements in Retrieval Augmented Generation Technology Since 2023', 'url': 'https://www.linkedin.com/pulse/latest-advancements-retrieval-augmented-generation-since-trung-dao-sfdvc', 'content': '### References\n\n NVIDIA. (2024). 17 Predictions for 2024: From RAG to Riches to Beatlemania and National Treasures. Retrieved from \n TechTarget. (2023). 9 top AI and machine learning trends. Retrieved from \n Datanami. (2023). Taking GenAI from Good to Great: Retrieval-Augmented Generation and Real-Time Data. Retrieved from \n Perficient. (2023). Retrieval Augmented Generation: How Businesses Benefit from Dynamic Adaptation. Retrieved from \n arXiv. (2023). Acti

### Exercise 1: AI in Healthcare

Goal: Investigate and summarize the latest advancements in generative AI applied to healthcare in 2025.

- Design a prompt that asks the agent to retrieve the most recent updates.
- Ensure the agent outputs a structured response in Markdown.

In [17]:
# Your prompt here
prompt_1 = """Summarize the latest advancements of generative AI in healthcare: 
Main heading: Generative AI in healthcare 
subheading per key model developement 
- Bullet points for features
- Source links if any
Keep it concise.
"""


# Run it
response_1 = agent_executor({"input":prompt_1})




> Entering new AgentExecutor chain...
I need to gather the latest information on advancements in generative AI specifically in the healthcare sector. This will involve searching for recent developments, key models, and their features. 

Action: TavilySafeSearch  
Action Input: "latest advancements of generative AI in healthcare 2023"  [{'title': 'Generative Artificial Intelligence Use in Healthcare - PMC - NIH', 'url': 'https://pmc.ncbi.nlm.nih.gov/articles/PMC11739231/', 'content': 'reveals that approximately 75% of large healthcare organizations are currently using or planning to scale up generative AI in their operations . In 2023, Gen AI technologies made substantial advancements, as evidenced by a recent study in which ChatGPT surpassed physicians in both quality and empathy metrics , and Google’s MedPaLM-2 LLM achieved expert-level scores on the United States Medical Licensing Examination . Furthermore, the first drug entirely designed with Gen AI entered clinical trials , whil

In [26]:
from IPython.display import Markdown, display
display(Markdown(str(response_1)))


{'input': 'Summarize the latest advancements of generative AI in healthcare: \nMain heading: Generative AI in healthcare \nsubheading per key model developement \n- Bullet points for features\n- Source links if any\nKeep it concise.\n', 'output': '**\n\n### Generative AI in Healthcare\n\n#### Key Model Developments\n\n- **ChatGPT and MedPaLM-2**\n  - ChatGPT has surpassed physicians in quality and empathy metrics.\n  - MedPaLM-2 achieved expert-level scores on the US Medical Licensing Examination.\n  - Source: [PMC - NIH](https://pmc.ncbi.nlm.nih.gov/articles/PMC11739231/)\n\n- **FunctionalMind**\n  - An AI system that assists clinicians by analyzing medical knowledge bases and summarizing clinical research.\n  - Aims to provide actionable insights for patient care.\n  - Source: [John Snow Labs](https://www.johnsnowlabs.com/generative-ai-healthcare/)\n\n- **Synthetic Data Generation**\n  - Techniques like GANs and VAEs are used to create synthetic healthcare records, enhancing predictive models for diabetes and other conditions.\n  - Source: [PMC - NIH](https://pmc.ncbi.nlm.nih.gov/articles/PMC12620437/)\n\n- **Generative Adversarial Networks (GANs)**\n  - Used for generating high-quality medical images, improving diagnostic capabilities in areas like dermatology and radiology.\n  - Source: [PMC - NIH](https://pmc.ncbi.nlm.nih.gov/articles/PMC12620437/)\n\n- **Natural Language Processing (NLP) Enhancements**\n  - New generative AI models improve natural language generation, summarization, and insight retrieval, enhancing operational efficiencies in healthcare.\n  - Source: [Deloitte US](https://www.deloitte.com/us/en/Industries/life-sciences-health-care/articles/generative-ai-in-healthcare.html)\n\nThese advancements indicate a significant shift towards integrating generative AI in healthcare, with potential benefits in patient care, operational efficiency, and medical research.'}

### Exercise 2: AI Startups Landscape

Goal: Track 2025’s top emerging AI startups and their innovations.

- Create a prompt that instructs the agent to deliver a clean Markdown summary.
- Tip: Ask for company names, product highlights, and sources.

In [21]:
# Your prompt here
prompt_2 = """Summarize the AI startups Landscape: 
Main heading: Give all the details of the Startups and their main objective
subheading per key model developement 
- Bullet points for their target and goals their workforce, vision
- Source links if any
Keep it concise.
"""

# Run it
response_2 = agent_executor({'input': prompt_2})




> Entering new AgentExecutor chain...
To summarize the AI startups landscape, I need to gather information on various AI startups, their objectives, target markets, workforce, and vision. This will require a web search to find up-to-date and relevant information.

Action: TavilySafeSearch
Action Input: "AI startups landscape 2023 summary"
[{'title': 'State of AI 2023 Report', 'url': 'https://www.cbinsights.com/research/report/ai-trends-2023/', 'content': "However, the median deal size has also held steady YoY at $4.4M, suggesting that investor attention is spread across AI startups of all sizes and stages.\n\nWe’ve even seen early-stage rounds soar past the $100M+ mark — in Q4’23, 3 startups raised Series A mega-rounds:\n\nNotably, all 3 of these companies develop open-source models for generative AI. The deals come at a time when open-source models are closing in on closed-source ones in terms of performance. \n\nA wave of M&A sweeps across AI\n\nA wave of M&A sweeps across AI [...]

In [25]:
display(Markdown(str(response_2)))


{'input': 'Summarize the AI startups Landscape: \nMain heading: Give all the details of the Startups and their main objective\nsubheading per key model developement \n- Bullet points for their target and goals their workforce, vision\n- Source links if any\nKeep it concise.\n', 'output': 'The AI startups landscape in 2023 features key players like OpenAI, Anthropic, Mistral AI, Jasper.ai, and xAI, each with distinct objectives and visions. The market is marked by substantial funding, particularly in generative AI, and a competitive environment driven by investments from major tech companies.'}

### Exercice 3: Compare Two Tech Products
- **Prompt idea:** Compare the key features, pricing, and reviews of OpenAI’s ChatGPT Team and Anthropic’s Claude Pro.
- Ensure the agent outputs a structured response in Markdown.




In [22]:
# Your prompt here
prompt_3 = """Compare the OpenAI's ChatGPT vs Anthropic's Claude Pro: 
Main heading: Give all the details of the comparison of ChatGPT vs Claude Pro only 
- Bullet points for their comparisons and give specifically the field in one is better than the other
Keep it concise.
"""

# Run it
response_3 = agent_executor({'input': prompt_3})




> Entering new AgentExecutor chain...
I need to gather information on the comparison between OpenAI's ChatGPT and Anthropic's Claude Pro. This will involve looking for specific details about their features, strengths, and weaknesses in various fields. 

Action: TavilySafeSearch  
Action Input: "ChatGPT vs Claude Pro comparison 2023"  [{'title': 'Claude AI vs ChatGPT: A Practical Comparison', 'url': 'https://www.appypieautomate.ai/blog/claude-vs-chatgpt', 'content': 'Web Browsing and Real-Time Information: ChatGPT has an optional browsing mode (for Plus users, with GPT-4 Browsing). This means if you ask a question about current events or something on the internet, ChatGPT can actually perform a web search and read content from the web to provide you an answer. For example, you can ask “What’s the latest NASA announcement today?” and ChatGPT can go online, find the information, and summarize it. This capability was turned off and on a few times as they refined it (due to issues like re

In [24]:
display(Markdown(str(response_3)))


{'input': "Compare the OpenAI's ChatGPT vs Anthropic's Claude Pro: \nMain heading: Give all the details of the comparison of ChatGPT vs Claude Pro only \n- Bullet points for their comparisons and give specifically the field in one is better than the other\nKeep it concise.\n", 'output': "The comparison between OpenAI's ChatGPT and Anthropic's Claude Pro highlights that ChatGPT excels in real-time information access and multimodal capabilities, while Claude Pro is superior in long-context reasoning and safety-focused outputs."}

## Bonus Task: Propose and Implement Your Own Use Case

As a final challenge, think of a real-world scenario where an AI agent could provide value using web search or external tools.
